# 08 · Agents as files – declare, review, deploy

Agents created in a notebook are hard to review and repeat. Put the definition in a **file in git**, deploy it from CI, get a new version each time. Three ways to do that with Foundry:

| 📄 Route | 🧰 Tool | 🔎 What lives in git |
|---|---|---|
| **Our YAML + SDK** (this chapter) | `scripts/deploy_agents.py` | `agents/*.yaml` |
| **azd `azure.yaml` services** | `azd deploy` | `azure.ai.agent` service (`kind: prompt` or `hosted`), toolboxes, skills, connections |
| **Portal export** | Foundry portal | copy the YAML/JSON view as a starting point |

> ℹ️ The older `agent.yaml` / `agent.manifest.yaml` files are deprecated: current `azd` keeps agent configuration inside `azure.yaml`. Deep dive: [docs/agents-yaml.md](../docs/agents-yaml.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

print((ROOT / "agents" / "jarvis.yaml").read_text())
print("--- instructions_file ---")
print((ROOT / "agents" / "jarvis.instructions.md").read_text())

## 1 · Dry run
`${VARS}` are resolved from `.env`; nothing is sent to Foundry. Use this in a pull-request check.

In [ ]:
sh("python scripts/deploy_agents.py agents/jarvis.yaml --dry-run")

## 2 · Deploy
Same name = **new version**, so re-running is safe.

In [ ]:
sh("python scripts/deploy_agents.py agents/jarvis.yaml")
r = ask_agent("jarvis", "What is 17 * 23 * 5? Use your tools.")
print(r.output_text, tool_calls(r))

## 3 · The azd route
`azd` can scaffold the same thing so you do not hand-write the YAML. In an **empty scratch folder**:

```bash
azd ai agent init jarvis-azd --kind prompt --agent-name jarvis-azd \
    --project-id <project resource id> --model gpt-5-6-terra \
    --instructions "You are J.A.R.V.I.S."
```

It writes an `azure.yaml` with an `azure.ai.agent` service (`kind: prompt`, model, instructions). `azd deploy` creates a new agent version; `azd ai agent show` and `azd ai agent invoke` inspect and call it. Compare the generated file with [agents/jarvis.yaml](../agents/jarvis.yaml): the ideas are the same, the schema is owned by `azd`.

This repo's own [azure.yaml](../azure.yaml) already uses that model for the hosted agent (chapter 09).

## 👀 In the portal
**Build → Agents**: every YAML you deployed is an agent, and a re-deploy adds a new version.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
delete_agent("jarvis")

## 🎯 Recap
| Need | Use |
|---|---|
| Small team, any CI, prompt agents | `agents/*.yaml` + `deploy_agents.py` |
| One tool for infra + agents + toolboxes + skills | azd `azure.yaml` services |

> 💡 Next: [09 · Hosted agent](09_hosted_agent.ipynb) – your own code, Foundry's runtime.